# Tech Challenge - Fase 1: Diagnóstico de Câncer de Mama
Este notebook atende aos requisitos do Tech Challenge Fase 1, abordando a classificação de dados médicos utilizando algoritmos de Machine Learning. Foram adicionadas análises exploratórias avançadas, matriz de correlação, explicabilidade do modelo com SHAP, gráfico de importância das variáveis, resumos explicativos ao final de cada etapa, discussão crítica para a prática médica e exportação do modelo.

In [ ]:
# ==========================================
# 1. INSTALAÇÃO DE PACOTES NECESSÁRIOS
# ==========================================

# Instala a biblioteca SHAP (interpretabilidade e explicabilidade do modelo) e garante a versão do joblib.
!pip install shap joblib -q

In [ ]:
# ==========================================
# 2. IMPORTAÇÕES BÁSICAS E CONFIGURAÇÕES
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import logging
import warnings

# Supressão de avisos do sistema e do SHAP para manter o notebook limpo e legível
warnings.filterwarnings('ignore')
logging.getLogger('shap').setLevel(logging.ERROR)

# Importações do Scikit-Learn para dados, métricas e pré-processamento
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# Importações de Algoritmos de Classificação
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

> 📌 **Resumo Explicativo (Step 1 & 2: Setup e Importações)**
> * **O que foi feito:** O ambiente Python foi configurado com todas as bibliotecas necessárias para manipulação de dados, modelagem preditiva e explicabilidade com SHAP.
> * **Conclusão Técnica:** O ambiente está pronto para processamento com supressão de warnings para visualização limpa.
> * **Próximo Passo:** Carregar o dataset médico Wisconsin Breast Cancer para início da Análise Exploratória.

In [ ]:
# ==========================================
# 3. EXPLORAÇÃO DE DADOS: CARGA INICIAL
# ==========================================

# Carregamos o dataset Wisconsin Breast Cancer do scikit-learn.
# A base contém 569 amostras e 30 atributos citológicos numéricos.
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)

# Mapeamento da variável alvo (Target): 0 = Maligno (Doença), 1 = Benigno (Saudável)
df['target'] = data.target

print(f"Formato do dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas.\n")
display(df.head())

> 📌 **Resumo Explicativo (Step 3: Carga do Dataset)**
> * **O que os dados mostram:** O dataset é composto por 569 observações e 31 colunas (30 atributos biomédicos + 1 variável alvo `target`).
> * **Conclusão Técnica:** Não há dados nulos aparentes na carga inicial, e a estrutura tabular encontra-se no formato esperado.
> * **Próximo Passo:** Avaliar as estatísticas descritivas e a matriz de correlação das variáveis.

In [ ]:
# ==========================================
# 4. EXPLORAÇÃO DE DADOS: ESTATÍSTICAS E CORRELAÇÃO
# ==========================================

# Análise das estatísticas descritivas básicas para entender escalas, médias e desvios padrão.
print("=== Estatísticas Descritivas ===")
display(df.describe())

# Geração do Heatmap de Correlação para identificar multicolinearidade entre os atributos biomédicos.
plt.figure(figsize=(20, 15))
correlacao = df.corr()
mask = np.triu(np.ones_like(correlacao, dtype=bool))
sns.heatmap(correlacao, mask=mask, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Matriz de Correlação das Variáveis Médicas', fontsize=18)
plt.show()
plt.close()

> 📌 **Resumo Explicativo (Step 4: Estatísticas e Matriz de Correlação)**
> * **O que os dados mostram:** Atributos relacionados a dimensões celulares (como `mean radius`, `mean perimeter` e `mean area`) possuem forte correlação positiva entre si e alta correlação negativa com a variável alvo.
> * **Conclusão Técnica:** Existe alta multicolinearidade e grande diferença de escala entre as variáveis (ex: desvios entre valores de milésimos e milhares).
> * **Próximo Passo:** Visualizar o Pairplot das principais variáveis e aplicar a padronização no pré-processamento.

In [ ]:
# ==========================================
# 4.1 GRÁFICOS DE RELAÇÃO: TOP VARIÁVEIS
# ==========================================

# Visualização da dispersão e separabilidade das principais variáveis de maior impacto clínico.
top_features = ['mean radius', 'mean perimeter', 'mean area', 'worst area', 'worst concave points']

print("Gerando Pairplot para as principais variáveis...")
g = sns.pairplot(df[top_features + ['target']], hue='target', palette='Set1', diag_kind='kde', corner=True)
g.fig.suptitle('Relação entre as Principais Variáveis Preditoras', y=1.02, fontsize=16)
plt.show()
plt.close()

> 📌 **Resumo Explicativo (Step 4.1: Pairplot de Top Variáveis)**
> * **O que os dados mostram:** O gráfico de densidade (KDE) e dispersão revela uma nítida separação visual entre tumores Malignos (0) e Benignos (1) com base no tamanho celular (`mean area`, `worst concave points`).
> * **Conclusão Técnica:** Os dados possuem forte poder discriminatório linear e não-linear para modelos de classificação.
> * **Próximo Passo:** Dividir o dataset em treino/teste e realizar a padronização das escalas.

In [ ]:
# ==========================================
# 5. PRÉ-PROCESSAMENTO
# ==========================================

# Divisão do dataset em matriz de características (X) e rótulos da classe alvo (y)
X = df.drop('target', axis=1)
y = df['target']

# Separação entre dados de Treino (80%) e Teste (20%).
# O parâmetro stratify=y mantém a distribuição proporcional das duas classes em ambos os conjuntos.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Aplicação do StandardScaler (Média=0, Desvio Padrão=1).
# A escala padronizada evita vieses e garante a convergência adequada de algoritmos sensíveis como Regressão Logística, SVM e KNN.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Pré-processamento concluído: Variáveis separadas, divididas e padronizadas na mesma escala.")

> 📌 **Resumo Explicativo (Step 5: Pré-Processamento e Escalonamento)**
> * **O que foi feito:** O dataset foi dividido em 80% treino e 20% teste com estratificação da classe alvo, seguido de padronização por `StandardScaler`.
> * **Conclusão Técnica:** Evitou-se o vazamento de dados (*data leakage*) ajustando o scaler apenas no conjunto de treino.
> * **Próximo Passo:** Treinar e comparar múltiplos algoritmos de classificação.

In [ ]:
# ==========================================
# 6. MODELAGEM: TREINAMENTO E AVALIAÇÃO COMPARATIVA
# ==========================================

# Instanciação de modelos de diferentes famílias (lineares, árvores e distâncias)
modelos = {
    "Regressão Logística": LogisticRegression(random_state=42, max_iter=500),
    "Árvore de Decisão": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier()
}

resultados = []
print("Iniciando treinamento e avaliação dos modelos...")
for nome, modelo in modelos.items():
    modelo.fit(X_train_scaled, y_train)
    y_pred = modelo.predict(X_test_scaled)
    resultados.append({
        "Modelo": nome,
        "Acurácia": accuracy_score(y_test, y_pred),
        "Recall (Sensibilidade)": recall_score(y_test, y_pred),
        "Precisão": precision_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })

# Ranqueamento dos modelos ordenado pelo F1-Score (balanceamento harmônico entre Precisão e Recall)
df_resultados = pd.DataFrame(resultados).sort_values(by="F1-Score", ascending=False)
for col in df_resultados.columns[1:]:
    df_resultados[col] = df_resultados[col].apply(lambda x: f"{x*100:.2f}%")

display(df_resultados.style.hide(axis="index"))

> 📌 **Resumo Explicativo (Step 6: Comparação de Modelos)**
> * **O que os dados mostram:** A **Regressão Logística** e o **SVM** alcançaram o topo do ranqueamento com **98.25% de Acurácia** e **98.61% de F1-Score/Recall**.
> * **Conclusão Técnica:** A Regressão Logística é a escolha ideal pela excelente performance combinada à alta interpretabilidade e baixo custo computacional.
> * **Próximo Passo:** Analisar a Matriz de Confusão detalhada do modelo campeão.

In [ ]:
# ==========================================
# 7. MATRIZ DE CONFUSÃO DO MODELO CAMPEÃO
# ==========================================

# Selecionado a Regressão Logística devido ao melhor desempenho conjunto de F1-Score, Acurácia e Recall (98.61%)
melhor_modelo = modelos["Regressão Logística"]
y_pred_melhor = melhor_modelo.predict(X_test_scaled)

# Matriz de Confusão do modelo vencedor no conjunto de testes
cm = confusion_matrix(y_test, y_pred_melhor)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Maligno (0)', 'Benigno (1)'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Matriz de Confusão - Regressão Logística')
plt.show()
plt.close()

print("\n📋 Relatório Completo de Classificação por Classe:")
print(classification_report(y_test, y_pred_melhor, target_names=['Maligno (0)', 'Benigno (1)']))

> 📌 **Resumo Explicativo (Step 7: Matriz de Confusão e Relatório por Classe)**
> * **O que os dados mostram:** O modelo errou apenas 2 amostras de 114 do conjunto de teste (41 acertos de 42 para casos malignos e 71 acertos de 72 para benignos).
> * **Conclusão Técnica:** O modelo apresentou um nível extremamente baixo de Falsos Negativos (1 caso), o que é essencial na área médica.
> * **Próximo Passo:** Analisar quais variáveis mais influenciaram essa decisão utilizando Feature Importance e SHAP.

In [ ]:
# ==========================================
# 8. IMPORTÂNCIA DAS VARIÁVEIS E EXPLICABILIDADE (SHAP)
# ==========================================

# 8.1 Gráfico de Importância Nativa das Variáveis (Top 10 do modelo Regressão Logística)
if hasattr(melhor_modelo, 'feature_importances_'):
    importances = melhor_modelo.feature_importances_
else:
    importances = np.abs(melhor_modelo.coef_[0])

# Ordenação decrescente exata das 10 variáveis de maior magnitude/coeficiente
indices = np.argsort(importances)[::-1][:10]
top_10_features = [X.columns[i] for i in indices]
top_10_importances = importances[indices]

plt.figure(figsize=(10, 5))
plt.bar(top_10_features, top_10_importances, color='#238082')
plt.xticks(rotation=45, ha='right')
plt.title('Top 10 Variáveis Mais Importantes (Feature Importance Nativas)')
plt.tight_layout()
plt.show()
plt.close()

# 8.2 Explicabilidade Global com a Biblioteca SHAP
print("Gerando mapa de explicabilidade global com SHAP...")
explainer = shap.LinearExplainer(melhor_modelo, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns, show=False)
plt.title('Importância e Impacto das Variáveis no Modelo (Regressão Logística)', fontsize=14)
plt.show()
plt.close()

> 📌 **Resumo Explicativo (Step 8: Explicabilidade e Importância das Variáveis)**
> * **O que os dados mostram:** Variáveis do grupo `worst` (como `worst texture`, `radius error`, `worst area` e `worst concave points`) possuem os maiores coeficientes e valores SHAP no modelo.
> * **Conclusão Técnica:** O modelo não é uma "caixa-preta"; ele toma decisões alinhadas com a literatura médica (alterações nas piores medições nucleares indicam malignidade).
> * **Próximo Passo:** Realizar a discussão crítica sobre a integração clínica e salvar os arquivos para deploy.

## 9. Discussão Crítica: O Modelo na Prática Médica

A aplicação de modelos preditivos na saúde feminina exige cuidados éticos e técnicos estritos. Respondendo aos requisitos do projeto:

**O modelo pode ser utilizado na prática?** Sim, mas exclusivamente operando como um **Sistema de Apoio à Decisão Clínica (CDSS - Clinical Decision Support System)**. Em nenhuma hipótese ele substitui o diagnóstico final de um patologista humano.

**Como ele seria integrado à realidade de um hospital?**
*   **Triagem e Priorização (Triage):** Em hospitais com alto volume de exames de biópsia para analisar, o algoritmo pode ser o primeiro a "ler" os dados do laboratório. Casos apontados pelo modelo como altamente passíveis de malignidade recebem um marcador (*flag*) urgente, indo para o topo da fila de laudos do médico. Isso acelera a identificação precoce e o início de tratamentos críticos.
*   **Dupla Checagem (Second Opinion Automatizada):** Após o médico redigir o laudo de um exame visualmente, ele confronta seu diagnóstico com o sistema. Se houver divergência forte (Ex: o patologista considerou *Benigno*, mas a IA indicou 98% de ser *Maligno*), o médico possui um gatilho seguro para reavaliar a lâmina sob o microscópio antes de liberar o laudo final para a paciente.

**O papel imprescindível do Médico e da Explicabilidade:** O gráfico gerado pelo método **SHAP** (acima) é a peça chave desta aplicação. A IA não está dizendo cegamente "é câncer". O SHAP diz ao médico: *"Estou indicando chance de ser maligno porque as medidas de área ('worst area'), o perímetro do núcleo celular ('worst perimeter') e o número de pontos côncavos no contorno da célula ('worst concave points') estão extremamente altos nestes dados específicos"*.

Dessa maneira, o médico consegue validar clinicamente a lógica matemática do modelo e mantém absoluta autoridade na decisão clínica final, aliando o ganho de produtividade da tecnologia com a necessária responsabilização e segurança no trato do câncer feminino.

> 📌 **Resumo Explicativo (Step 9: Aplicação Clínica)**
> * **O que os dados mostram:** A alta acurácia e transparência explicativa viabilizam o uso como suporte e triagem urgente no ambiente hospitalar.
> * **Conclusão Técnica:** O algoritmo agrega agilidade e segurança sem violar a soberania do diagnóstico do médico especialista.
> * **Próximo Passo:** Exportar os arquivos para o ambiente de produção.

In [ ]:
# ==========================================
# 10. SALVAMENTO DO MODELO E SCALER PARA API
# ==========================================

nome_arquivo_modelo = 'modelo_breast_cancer.pkl'
nome_arquivo_scaler = 'scaler_breast_cancer.pkl'

# Exportação dos artefatos salvos com joblib para uso futuro em uma API REST de produção
joblib.dump(melhor_modelo, nome_arquivo_modelo)
joblib.dump(scaler, nome_arquivo_scaler)

print(f"✅ Arquivos salvos com sucesso!")
print(f"Modelo salvo como: {nome_arquivo_modelo}")
print(f"Scaler salvo como: {nome_arquivo_scaler}")

> 📌 **Resumo Explicativo (Step 10: Exportação de Artefatos)**
> * **O que foi feito:** O modelo treinado de Regressão Logística e o pré-processador `StandardScaler` foram serializados em formato `.pkl`.
> * **Conclusão Técnica:** O projeto está 100% concluído e pronto para deploy no backend da API.